# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (sMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on PyMC.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianLogisticRegression, BnnLayerParams, BnnParams, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.generics:GenericModel` has been moved to `pydantic.BaseModel`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[ 0.74940531  0.51642045  0.42028269 -0.80737275  0.9048318 ]
 [-0.34920399  0.51882517 -0.22286712 -0.72928127  0.76306319]
 [-0.45620418 -0.34757661 -0.0056537  -0.70585844 -0.97081006]
 [ 0.12822752  0.11113249  0.02538047  0.77718962 -0.23429496]
 [-0.08248037  0.53110768  0.08511385  0.16207921 -0.24445767]
 [-0.68288796  0.73097275  0.41594788 -0.20976203  0.21360162]
 [-0.45841675  0.87810032 -0.51543081  0.8769486   0.99734166]
 [-0.60821195 -0.58293135 -0.60925256 -0.64405144  0.27757592]
 [ 0.22833159  0.1387103  -0.31531232 -0.65496029  0.41923547]
 [ 0.39300462 -0.96855747 -0.90259842 -0.48682849 -0.81193479]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])

actions = {
    "a1": BayesianLogisticRegression(model_params=model_params),
    "a2": BayesianLogisticRegression(model_params=model_params),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a1', 'a1', 'a2', 'a1', 'a1', 'a1', 'a1', 'a2', 'a2', 'a1']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [1, 1, 0, 0, 1, 0, 1, 1, 1, 0]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/pytensor/link/c/cmodule.py:2968: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(


2025-07-16 10:34:56.549 | ERROR    | pymc.stats.convergence:log_warning:181 - There were 1000 divergences after tuning. Increase `target_accept` or reparameterize.


2025-07-16 10:34:56.550 | ERROR    | pymc.stats.convergence:log_warning:181 - The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
